# Auto-Label Continuous Signal Data for MPLAB ML Development Suite

This notebook automatically labels continuous discrete signals for use in MPLAB ML Development Suite.

## About This Notebook

Consider an ML model developed for predictive maintenance based on Quadrature Current (Iq), RPM or other parameters. In such cases, data can be collected as continuous signals. This notebook allows you to automatically label such data (using folder and file names) and uploads the labelled data to the cloud-based MPLAB ML Development Suite.


### Folder Setup

1.	Create a folder with the ML Development Suite project name (PROJECT_NAME).
2.	Inside this folder, create as many sub-folders as the labels taking care that each folder name should be the exact classifier label (labelA, labelB, etc).
3.	Collect sufficient data in CSV files for each classifier and store them in their respective folder. Name the CSV files in the format LABEL_N where N is an incremental number. Similarly, collect data in CSV files for testing the model. Name the CSV files in the format LABEL_tN where N is an incremental number.
```

PROJECT_NAME/
├── labelA/
│   ├── labelA_1.csv
│   ├── labelA_2.csv
│   └── labelA_t1.csv
├── labelB/
│   └── labelB_1.csv
│   └── labelB_2.csv
│   └── labelB_t1.csv
├── labelB/
│   └── labelB_1.csv
│   └── labelB_2.csv
│   └── labelB_t1.csv
└── labelC-gesture/
    └── labelC.csv
```


### Workflow of this Notebook

1. **Upload Data** - Your recorded gesture CSV files
2. **Auto-Label** - Assign labels to the collected data based on folder names
3. **Create Metadata** - Assign metadata - Test/Train to captues based on filename
4. **Create Project** - Generate MPLAB ML-compatible files (.dclproj)
5. **Upload to MPLAB** - Train your model in MPLAB ML

---

## Table of Contents

**Section 1**: Setup and Data Upload

**Section 2**: Auto-Labeling based on folder name

**Section 3**: MPLAB ML Project Creation

**Section 4**: Upload to MPLAB ML

---

## Section 1: Setup and Data Upload

### 1.1 Import Required Libraries

We'll use standard Python libraries for data processing and the MPLAB ML SDK for project creation. The MPLAB ML SDK now supports Python 3.12, so we can use Colab's default Python environment.

In [1]:
# Standard library imports
import os
import glob
import json
import sys
from pathlib import Path

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Display versions
print("Environment Information:")
print(f"  Python: {sys.version.split()[0]}")
print(f"  Pandas: {pd.__version__}")
print(f"  NumPy: {np.__version__}")
print("\n✓ Libraries imported successfully")

Environment Information:
  Python: 3.12.12
  Pandas: 2.2.2
  NumPy: 2.0.2

✓ Libraries imported successfully


### 1.2 Upload Your Data

#### Data Organization

The root folder should be named after your project name. Example - PredictiveMaintenance. Your data should be organized in folders based on operation condition. For example - Normal (Normal load), unbalanced (unbalanced load). Each type of operation condition should have its own folder with CSV files containing the data. Train data filename should end with _N and test data filename should end with _tN (Where N is the file number).

**Expected folder structure:**
```
PredictiveMaintenance/
├──Normal/
│   ├── normal_1.csv
│   ├── normal_2.csv
│   └── normal_t1.csv
└──Unbalanced/
    └── unbalanced_1.csv
    └── unbalanced_2.csv
    └── unbalanced_t1.csv

```



**How to upload the Data**

To upload the data:
1. Compress your `project` folder containing training data to a ZIP file on your computer
2. Upload the ZIP file to the /content folder


### 1.3 Extract Your Gesture Data

In [2]:

import zipfile

# List available ZIP files
zip_files = glob.glob('/content/*.zip')

if zip_files:
    print(f"Found {len(zip_files)} ZIP file(s):")
    for zf in zip_files:
        print(f"  - {os.path.basename(zf)}")

    # Extract the first ZIP file found
    zip_path = zip_files[0]
    print(f"\nExtracting: {os.path.basename(zip_path)}")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        names = zip_ref.namelist()
        zip_ref.extractall('/content/')

    print("✓ Files extracted to /content/")
    # Detect top-level folder inside ZIP
    top_folders = set(name.split('/')[0] for name in names if '/' in name)

    if len(top_folders) == 1:
        # Normal case: ZIP contains one root folder
        folder_name = list(top_folders)[0]
        DATA_DIR = os.path.join('/content', folder_name)
    else:
        # ZIP contains loose files
        DATA_DIR = '/content'

    print("Dataset path detected:")
    print("Folder name:", folder_name)
    print(DATA_DIR)

else:
    print("No ZIP files found in /content/")
    print("If you uploaded individual files, you can skip this cell.")

Found 1 ZIP file(s):
  - PrediciveMaintWedge.zip

Extracting: PrediciveMaintWedge.zip
✓ Files extracted to /content/
Dataset path detected:
Folder name: PrediciveMaintWedge
/content/PrediciveMaintWedge


### 1.4 Verify Data Upload

Let's check that your data was uploaded correctly and is in the expected location.

In [3]:
# List all subdirectories (class folders)
class_folders = [d for d in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, d))]

print(f"Found {len(class_folders)} class folder(s):")

total_files = 0
for folder in sorted(class_folders):
    folder_path = os.path.join(DATA_DIR, folder)
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
    total_files += len(csv_files)
    print(f"\n  📁 {folder}/")
    for csv_file in sorted(csv_files):
        file_size = os.path.getsize(csv_file) / 1024  # KB
        print(f"     - {os.path.basename(csv_file):30s} ({file_size:>8.1f} KB)")

print(f"\n{'='*60}")
print(f"Total: {total_files} CSV file(s) ready for processing")
print(f"{'='*60}")



Found 2 class folder(s):

  📁 Normal/
     - normal_1.csv                   (    69.0 KB)
     - normal_2.csv                   (    69.3 KB)
     - normal_3.csv                   (    69.1 KB)
     - normal_4.csv                   (    69.2 KB)
     - normal_5.csv                   (    69.7 KB)
     - normal_6.csv                   (    69.6 KB)
     - normal_t1.csv                  (    69.8 KB)
     - normal_t2.csv                  (    69.1 KB)

  📁 Unbalanced/
     - unbalanced_1.csv               (    69.6 KB)
     - unbalanced_2.csv               (    69.7 KB)
     - unbalanced_3.csv               (    69.7 KB)
     - unbalanced_4.csv               (    69.8 KB)
     - unbalanced_5.csv               (    69.8 KB)
     - unbalanced_6.csv               (    69.7 KB)
     - unbalanced_t1.csv              (    69.4 KB)
     - unbalanced_t2.csv              (    69.8 KB)

Total: 16 CSV file(s) ready for processing


In [4]:
# Find all CSV files
all_csv_files = []
for classifier_folder in sorted(os.listdir(DATA_DIR)):
    folder_path = os.path.join(DATA_DIR, classifier_folder)
    if os.path.isdir(folder_path):
        csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
        all_csv_files.extend(csv_files)

if not all_csv_files:
    print("No CSV files found!")
else:
    # Load the first CSV file as a sample
    sample_file = all_csv_files[0]
    df_sample = pd.read_csv(sample_file)

    print(f"Sample file: {os.path.basename(sample_file)}")
    print(f"Full path: {sample_file}")
    print(f"\nDataset shape: {df_sample.shape[0]:,} rows × {df_sample.shape[1]} columns")
    print(f"\nColumn names:")
    for i, col in enumerate(df_sample.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nData types:")
    print(df_sample.dtypes)

    print(f"\nFirst 10 rows:")
    print(df_sample.head(10))

Sample file: normal_4.csv
Full path: /content/PrediciveMaintWedge/Normal/normal_4.csv

Dataset shape: 10,000 rows × 2 columns

Column names:
  1. IQ
  2. RPM

Data types:
IQ     int64
RPM    int64
dtype: object

First 10 rows:
   IQ  RPM
0  18  900
1  11  900
2  13  900
3  11  900
4  29  900
5  11  900
6  20  900
7  30  900
8  11  900
9  20  900


## Section 2:  Auto-Labeling based on folder name

Now we will automatically label your collected data based on folder name. Each folder name represents a unique classifier which we will parse to determine the required label classifiers for the ML model.

### 2.1 Define classifier Labels

First, let us define the mapping between classifiers and numeric label IDs for MPLAB ML. The sub-folder name is used to generate the labels.

In [5]:
import re

print("Scanning folders inside:", DATA_DIR)

# Get only valid subfolders
subfolders = [
    f for f in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, f))
    and not f.startswith('.')  # ignore hidden folders
]

print("Found folders:", subfolders)

# Convert folder names to lowercase
labels = sorted(set([folder.lower() for folder in subfolders]))

print("Extracted labels:", labels)

# Create label mapping
LABEL_MAP = {label: idx for idx, label in enumerate(labels)}

# Reverse mapping
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}

# Print mapping
print("\n Label Mapping:")
print("="*20)
for classifier, label_id in LABEL_MAP.items():
    print(f"  {classifier:10s} -> {label_id}")

Scanning folders inside: /content/PrediciveMaintWedge
Found folders: ['Unbalanced', 'Normal']
Extracted labels: ['normal', 'unbalanced']

 Label Mapping:
  normal     -> 0
  unbalanced -> 1


### 2.2 Labeling Functions

These functions will:
1. Extract classifier name from filename (e.g., "normal_1.csv" -> "normal")
2. Label all rows in CSV file with the mapping already performed for the label/ classifier.

In [6]:
def extract_classifier_name(filename):
    """
    Extract label name from filename.
    Examples: 'normal_1.csv' -> 'normal', 'normal/normal_t1.csv' -> 'normal'
    """
    basename = os.path.basename(filename)
    name = os.path.splitext(basename)[0]

    # Extract first part before '_'
    if '_' in name:
        classifier_name = name.split('_')[0]
    else:
        classifier_name = name.split('-')[0]

    return classifier_name.lower()


def label_classifier_data(df, classifier_name, label_map=LABEL_MAP):
    """
    Label CSV data with LABEL mapping.

    Args:
        df: DataFrame with data
        classifier_name: Classifier or Label (e.g., 'normal', 'unblanced')
        label_map: Dictionary mapping gesture names to label IDs

    Returns:
        DataFrame with added 'label' column
    """
    df = df.copy()
    if classifier_name not in LABEL_MAP:
        raise ValueError(f"Classifier '{classifier_name}' not found in LABEL_MAP")
    label_id = LABEL_MAP[classifier_name]
    # Assign label to every row
    df["label"] = label_id



    return df


print("Labeling functions defined")

Labeling functions defined


### 2.3 Process All Gesture Files

Now we'll process each CSV file, label the segments, and save the results.

In [7]:
# Create output directory for labeled files
LABELED_DIR = '/content/labeled_data'
os.makedirs(LABELED_DIR, exist_ok=True)

# Process each CSV file
print("Processing CSV files...")
print("="*70)

labeled_files = []
total_files = 0


for csv_file in sorted(all_csv_files):
    # Extract classifier name
    classifier_name = extract_classifier_name(csv_file)

    # Load data
    df = pd.read_csv(csv_file)
    #print(os.path.basename(csv_file), len(df))
    # Label the data
    df_labeled = label_classifier_data(df, classifier_name)

    # Create output filename
    basename = os.path.basename(csv_file)
    output_filename = f"{classifier_name}_labeled_{basename}"
    output_path = os.path.join(LABELED_DIR, output_filename)

    # Save labeled data
    df_labeled.to_csv(output_path, index=False)
    labeled_files.append(output_path)

    total_files += 1

    # Print summary
    label_counts = df_labeled['label'].value_counts().sort_index()
    print(f"\n{basename}")
    print(f"  Label: {classifier_name}")
    print(f"  Total samples: {len(df_labeled):,}")

    for label_id, count in label_counts.items():
        label_name = LABEL_NAMES.get(label_id, f"Unknown({label_id})")
        pct = 100.0 * count / len(df_labeled)
        print(f"    {label_name:10s}: {count:6,} samples ({pct:5.1f}%)")

print(f"\n{'='*70}")
print(f"Processing complete!")
print(f"  Files processed: {total_files}")
print(f"  Labeled files saved to: {LABELED_DIR}")

Processing CSV files...

normal_1.csv
  Label: normal
  Total samples: 10,000
    normal    : 10,000 samples (100.0%)

normal_2.csv
  Label: normal
  Total samples: 10,000
    normal    : 10,000 samples (100.0%)

normal_3.csv
  Label: normal
  Total samples: 10,000
    normal    : 10,000 samples (100.0%)

normal_4.csv
  Label: normal
  Total samples: 10,000
    normal    : 10,000 samples (100.0%)

normal_5.csv
  Label: normal
  Total samples: 10,000
    normal    : 10,000 samples (100.0%)

normal_6.csv
  Label: normal
  Total samples: 10,000
    normal    : 10,000 samples (100.0%)

normal_t1.csv
  Label: normal
  Total samples: 10,000
    normal    : 10,000 samples (100.0%)

normal_t2.csv
  Label: normal
  Total samples: 10,000
    normal    : 10,000 samples (100.0%)

unbalanced_1.csv
  Label: unbalanced
  Total samples: 10,000
    unbalanced: 10,000 samples (100.0%)

unbalanced_2.csv
  Label: unbalanced
  Total samples: 10,000
    unbalanced: 10,000 samples (100.0%)

unbalanced_3.csv


### 2.4 Verify Labeled Data

Let's verify the labeled data by checking one file.

In [8]:
# Check one labeled file
if labeled_files:
    sample_labeled = pd.read_csv(labeled_files[0])

    print(f"Sample labeled file: {os.path.basename(labeled_files[0])}")
    print(f"Columns: {list(sample_labeled.columns)}")
    print(f"\nLabel distribution:")

    for label_id, count in sample_labeled['label'].value_counts().sort_index().items():
        label_name = LABEL_NAMES[label_id]
        print(f"  {label_name}: {count:,} samples")

    print(f"\nFirst few rows:")
    print(sample_labeled[sample_labeled.columns].head(10))

Sample labeled file: normal_labeled_normal_1.csv
Columns: ['IQ', 'RPM', 'label']

Label distribution:
  normal: 10,000 samples

First few rows:
   IQ  RPM  label
0  22  900      0
1  37  900      0
2  20  900      0
3  19  900      0
4   1  900      0
5  14  900      0
6  13  900      0
7  28  900      0
8   2  900      0
9   5  900      0


## Section 3: Create MPLAB ML Project Files

Now we'll create the `.dclproj` manifest file that MPLAB ML uses to organize labeled data. This file describes:
- Which CSV files contain your data
- How the data is segmented (labeled regions)
- Metadata to set dataset type as test/ train

The .dclproj format allows MPLAB ML to understand the structure of your labeled data without having to manually label using the GUI.

### 3.1 Extract Labeled Segments

For the .dclproj file, we need to identify contiguous regions of each label. Let us define a helper function for it.

In [9]:
def extract_segments_for_dclproj(df):
    """
    Extract contiguous labeled segments for .dclproj format.

    Returns list of segments:
    [
      {"start": 0, "end": 100, "name": "Label", "value": "horizontal"},
      {"start": 101, "end": 200, "name": "Label", "value": "unknown"},
      ...
    ]
    """
    segments = []

    if 'label' not in df.columns:
        return segments

    # Find where label changes
    label_changes = df['label'].ne(df['label'].shift()).cumsum()

    for group_id in label_changes.unique():
        group_mask = label_changes == group_id
        group_indices = df[group_mask].index

        start_idx = int(group_indices[0])
        end_idx = int(group_indices[-1])
        label_id = int(df.loc[start_idx, 'label'])
        label_name = LABEL_NAMES.get(label_id, 'unknown')

        segments.append({
            "name": "Label",
            "value": label_name,
            "start": start_idx,
            "end": end_idx,
        })

    return segments

print("Segment extraction function defined")

Segment extraction function defined


Here we define another helper function to check if the CSV filename has a _t in it. Such data will be set with metadata "test". Otherwise it will be set to "train" dataset type for metadata.

In [10]:
# Helper function to check if this is test/ train data to set the metadata accordingly
def detect_dataset_type(filename):
    """
    Detect dataset type from filename.

    horizontal_1.csv   -> training
    horizontal_t1.csv  -> testing
    """
    basename = os.path.basename(filename).lower()

    if "_t" in basename:
        return "testing"
    else:
        return "training"

### 3.2 Create .dclproj Manifest

The manifest file lists all CSV files and their labeled segments.

In [11]:
def create_dclproj_manifest(labeled_csv_files, output_path):
    """
    Create .dclproj manifest file for MPLAB ML.

    Args:
        labeled_csv_files: List of paths to labeled CSV files
        output_path: Where to save the .dclproj file
    """
    manifest = []

    print("Creating manifest...")
    print("="*70)

    for csv_file in labeled_csv_files:
        basename = os.path.basename(csv_file)
        print(f"  Processing: {basename}")
        # Detect dataset type from filename
        dataset_type = detect_dataset_type(basename)
        # Load labeled data
        df = pd.read_csv(csv_file)

        # Extract segments
        segments = extract_segments_for_dclproj(df)
        print(f"    Found {len(segments)} segments")

        # Build manifest entry
        manifest.append({
            "file_name": basename,
            "metadata": [
                {"name": "dataset_type", "value": dataset_type},
            ],
            "sessions": [
                {
                    "session_name": "Session 1",
                    "segments": segments
                }
            ]
        })

    # Write manifest to file
    with open(output_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    print(f"\n{'='*70}")
    print(f"Manifest created: {output_path}")
    print(f"  Total files: {len(manifest)}")
    total_segments = sum(len(item['sessions'][0]['segments']) for item in manifest)
    print(f"  Total segments: {total_segments}")

    return output_path

print("Manifest creation function defined")

Manifest creation function defined


### 3.3 Generate Project Files

Let's create the .dclproj file and organize everything for upload.

In [12]:
# Configuration

#The project name is the root folder name obtained froms ection 1.3
PROJECT_NAME = folder_name
print("PROJECT_NAME:", PROJECT_NAME)
# Create MPLAB project directory
MPLAB_PROJECT_DIR = '/content/mplab_project'
os.makedirs(MPLAB_PROJECT_DIR, exist_ok=True)

# Copy labeled CSV files to project directory
print("Preparing MPLAB ML project files...")
print("="*70)

import shutil

mplab_csv_files = []
for labeled_file in labeled_files:
    basename = os.path.basename(labeled_file)
    dest_path = os.path.join(MPLAB_PROJECT_DIR, basename)
    shutil.copy2(labeled_file, dest_path)
    mplab_csv_files.append(dest_path)
    print(f"  Copied: {basename}")

# Create .dclproj manifest
manifest_filename = f"{PROJECT_NAME.lower()}.dclproj"
manifest_path = os.path.join(MPLAB_PROJECT_DIR, manifest_filename)

create_dclproj_manifest(
    mplab_csv_files,
    manifest_path)

print(f"\n{'='*70}")
print(f"MPLAB ML project ready!")
print(f"  Location: {MPLAB_PROJECT_DIR}")
print(f"  Manifest: {manifest_filename}")
print(f"  CSV files: {len(mplab_csv_files)}")

PROJECT_NAME: PrediciveMaintWedge
Preparing MPLAB ML project files...
  Copied: normal_labeled_normal_1.csv
  Copied: normal_labeled_normal_2.csv
  Copied: normal_labeled_normal_3.csv
  Copied: normal_labeled_normal_4.csv
  Copied: normal_labeled_normal_5.csv
  Copied: normal_labeled_normal_6.csv
  Copied: normal_labeled_normal_t1.csv
  Copied: normal_labeled_normal_t2.csv
  Copied: unbalanced_labeled_unbalanced_1.csv
  Copied: unbalanced_labeled_unbalanced_2.csv
  Copied: unbalanced_labeled_unbalanced_3.csv
  Copied: unbalanced_labeled_unbalanced_4.csv
  Copied: unbalanced_labeled_unbalanced_5.csv
  Copied: unbalanced_labeled_unbalanced_6.csv
  Copied: unbalanced_labeled_unbalanced_t1.csv
  Copied: unbalanced_labeled_unbalanced_t2.csv
Creating manifest...
  Processing: normal_labeled_normal_1.csv
    Found 1 segments
  Processing: normal_labeled_normal_2.csv
    Found 1 segments
  Processing: normal_labeled_normal_3.csv
    Found 1 segments
  Processing: normal_labeled_normal_4.csv
  

## Section 4: Upload to MPLAB ML
We are now ready to upload this project to MPLAB ML Developer Suite



### 4.1 Install MPLAB ML SDK
We'll install the SDK to use the APIs to connect to the cloud platform and upload the project.

In [13]:
# Install MPLAB ML SDK for project file creation
print("Installing MPLAB ML Development Suite SDK...")
!pip install mplabml --quiet

# Verify installation
try:
    import mplabml
    from mplabml.dclproj import dclproj
    print(f"✓ MPLAB ML SDK installed successfully")
    print(f"  Version: {mplabml.__version__}")
    print("\nReady to create MPLAB ML project files")
except ImportError as e:
    print(f"Error: {e}")
    print("Installation may have failed - please restart runtime and try again")

Installing MPLAB ML Development Suite SDK...
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.8/180.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.8 MB/s eta 0:00:00


✓ MPLAB ML SDK installed successfully
  Version: 2025.1.1

Ready to create MPLAB ML project files


You will be prompted to enter the API key generated in your MPLAB ML Development Suite account.

In [14]:
from mplabml import Client
from IPython.display import display, Javascript
# Force visible input prompt
api_key = input("🔑 Enter your MPLAB ML API Key: ").strip()
if not api_key:
    raise ValueError("API key is required!")

client = Client(api_key=api_key)
display(Javascript("alert('Connected to MPLAB ML Cloud successfully!');"))

🔑 Enter your MPLAB ML API Key: oTz40Oih.x6HElxOmJ7DMCO0gpq4iiMXhq2rGiDGG
Client connected to the cloud successfully


<IPython.core.display.Javascript object>

### 4.2 Upload the project to the cloud platform
We will now use the SDK API upload_project_dcli to upload the project manifest file we created in section 4.3.

In [15]:
OUTPUT_DIR = MPLAB_PROJECT_DIR
print("=" * 70)
print(f"UPLOADING PROJECT: {PROJECT_NAME}")
print("=" * 70)

print(f"\nManifest: {manifest_path}")
print(f"Upload directory: {OUTPUT_DIR}")

# Verify manifest exists
if not os.path.exists(manifest_path):
    raise FileNotFoundError(f"Manifest not found at: {manifest_path}")

# -------------------------------------------------
# Check if project exists and delete it
# -------------------------------------------------

print("\n🔍 Checking for existing project...")

try:
    projects_df = client.list_projects()

    if PROJECT_NAME in projects_df['Name'].values:
        print(f"⚠️ Project '{PROJECT_NAME}' already exists")
        print("🗑️ Deleting existing project...")
        client.delete_project(PROJECT_NAME)
        print("✓ Deleted successfully")
    else:
        print("✓ No existing project found")

except Exception as e:
    print(f"ℹ️ Could not check for existing project: {e}")

# -------------------------------------------------
# Upload Project
# -------------------------------------------------

print("\n⏳ Upload in progress...")
print("This may take 30-60 seconds...\n")

try:
    result = client.upload_project_dcli(
        name=PROJECT_NAME,
        dcli_path=manifest_path,
        force=True
    )

    print("\n" + "=" * 70)
    print("✅ UPLOAD COMPLETED SUCCESSFULLY!")
    print("=" * 70)
    print(f"\nProject '{PROJECT_NAME}' is now in MPLAB ML!\n")
    display(Javascript("alert(' Notebook executed successfully and your project is now uploaded!');"))
except Exception as e:
    print("\n" + "=" * 70)
    print("❌ UPLOAD FAILED")
    print("=" * 70)
    print(f"\nError: {e}\n")
    raise

UPLOADING PROJECT: PrediciveMaintWedge

Manifest: /content/mplab_project/predicivemaintwedge.dclproj
Upload directory: /content/mplab_project

🔍 Checking for existing project...
✓ No existing project found

⏳ Upload in progress...
This may take 30-60 seconds...

Project PrediciveMaintWedge does not exist, creating a new project.
Uploading Capture 0/16: normal_labeled_normal_1.csv
Segmenter Uploaded.
Uploading Capture 1/16: normal_labeled_normal_2.csv
Uploading Capture 2/16: normal_labeled_normal_3.csv
Uploading Capture 3/16: normal_labeled_normal_4.csv
Uploading Capture 4/16: normal_labeled_normal_5.csv
Uploading Capture 5/16: normal_labeled_normal_6.csv
Uploading Capture 6/16: normal_labeled_normal_t1.csv
Uploading Capture 7/16: normal_labeled_normal_t2.csv
Uploading Capture 8/16: unbalanced_labeled_unbalanced_1.csv
Uploading Capture 9/16: unbalanced_labeled_unbalanced_2.csv
Uploading Capture 10/16: unbalanced_labeled_unbalanced_3.csv
Uploading Capture 11/16: unbalanced_labeled_unbala

<IPython.core.display.Javascript object>

With this, you will have created a project in MPLAB ML Developer Suite cloud platform and uploaded all the collected and labelled data to it without having to manually add labels or metadata.